# Practical PyTorch · Part 4 — Companion Notebook

We open the black box by building it up: start with a single layer, stack a few into a tiny net, then read a real model (ResNet-18) and find it's the same idea at scale. We only *look* — no training.

## 1. The simplest model: a single nn.Linear

`nn.Linear` does a weighted sum: numbers in, multiply by learned weights, add a bias, numbers out. It's already a complete `nn.Module` — the same base class the giant models use.

In [ ]:
import torch
import torch.nn as nn

layer = nn.Linear(3, 2)   # 3 numbers in, 2 numbers out
print(layer)              # Linear(in_features=3, out_features=2, bias=True)

# the parameters — the actual numbers training would tune
print(layer.weight.shape)   # torch.Size([2, 3])
print(layer.bias.shape)     # torch.Size([2])

# a forward pass: numbers in the top, numbers out the bottom
x = torch.rand(1, 3)   # one example, 3 features
out = layer(x)
print(out.shape)       # torch.Size([1, 2])

## 2. Stack a few: a tiny network

`nn.Sequential` runs layers in order, top to bottom. `print(net)` shows a little tree — one line per layer.

In [ ]:
net = nn.Sequential(
    nn.Linear(10, 5),   # weighted sum: 10 numbers in, 5 out
    nn.ReLU(),          # keep the positives
    nn.Linear(5, 2),    # weighted sum: 5 numbers in, 2 out
)

print(net)

Feed it one random input and watch data flow through. The leading `1` in `torch.rand(1, 10)` is a *batch* dimension — a batch of one example.

In [ ]:
x = torch.rand(1, 10)   # one example, 10 features, random values
out = net(x)

print('input shape: ', tuple(x.shape))     # (1, 10)
print('output shape:', tuple(out.shape))   # (1, 2)
print('output:', out)

## 3. The common layers, in plain English

You've now used two of the three you'll meet everywhere — no math needed:

- **nn.Linear** — a weighted sum (the fully-connected layer; the last one turns internals into one score per category).
- **nn.Conv2d** — slides a small filter over an image, hunting for patterns. The workhorse of vision models.
- **nn.ReLU** — keep the positives: negatives become zero, positives pass through.

Others exist (`BatchNorm2d`, `MaxPool2d`, `Dropout`), but those three vibes carry you through Phase I.

## 4. Scale up: read a real model (ResNet-18)

A toy net is great for *feeling* how a model works. Now open a real, famous one — same idea, just bigger. `weights=ResNet18_Weights.DEFAULT` asks for the best official pretrained weights (downloaded once, then cached).

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

model = resnet18(weights=ResNet18_Weights.DEFAULT)

# same move as your tiny net — just print it and read the shape of it
print(model)

That indented outline *is* the tree — the same kind your tiny net had, just deeper. Spot the familiar faces: `Conv2d`, `ReLU`, `Sequential`, and a closing `Linear` (`fc`) producing one score per ImageNet category.

The parameters work the same way too — there are simply more of them. Count them, then peek at where the first few live.

In [ ]:
total = sum(p.numel() for p in model.parameters())
print(f'{total:,} parameters')   # 11,689,512 parameters  (your single Linear had 8)

for name, p in list(model.named_parameters())[:5]:
    print(name, tuple(p.shape))

The names mirror the tree — `layer1.0.conv1.weight` is just a path — and every parameter is a tensor with a shape, exactly like the `weight` and `bias` on your single `Linear`.

**Next: Part 5 — Running Your First Pretrained Model.**